# 🔎 UFVAI — Agente de IA para Pesquisa Científica

O **UFVAI** é um assistente de IA no Google Colab, sobre a arquitetura **OpenCode**, que automatiza o fluxo acadêmico — do levantamento bibliográfico à estruturação de artigos — com dados públicos do Brasil (**IBGE**, **DataSUS**, **INPE**, **CONAB**, **Banco Central** e outras).

| Benefício | Detalhe |
|-----------|---------|
| **100% Gratuito** | Infraestrutura em nuvem do Google |
| **Sem Instalação** | Pronto no navegador |
| **Foco Nacional** | **IBGE**, **DataSUS** e fontes oficiais BR |
| **Memória persistente** | Achados salvos automaticamente no seu Google Drive |
| **Base Tecnológica** | **OpenCode** |

---

### 🚀 Como Usar

1. **Ambiente de execução → Executar tudo** (`Ctrl+F9`).
2. Aguarde ~2 min e clique em **ABRIR O UFVAI**.
3. **O keep-alive fica ativo em background** (1 linha a cada 60s em `ufvai_keepalive.log`) — impede o encerramento por ociosidade com a aba minimizada. Limite honesto: o Colab pode destruir o runtime (~12 h de teto, cota). Se cair: **Runtime → Reiniciar sessão → Executar tudo**.
4. **Para encerrar por completo** (economiza cota): clique no menu **Ambiente de execução → Desconectar e ecluir o ambiente de execução**.

> **Primeira vez:** tela de Termos **v2.2** (`terms_version=6`) — informe **nome + e-mail** para ativação (LGPD art. 7º, V, com registro de IP). Estatísticas anônimas ficam ativas por padrão, sem cookies, e podem ser desmarcadas (opt-out).
> **Depois:** "Bem-vindo de volta" com nome persistido — clique em *Continuar*.

---

- **Drive:** `Meu Drive/PesquisAI` · **Navegador:** Chrome · **Versão:** v0.7.11
- **Registro:** SisPPG/UFV nº 10356285004 · **Site oficial:** https://der.ufv.br/ufvai/


In [ ]:
import os, sys, shutil, socket, subprocess
from IPython.display import display, HTML, update_display

REPO_URL = "https://github.com/gustavobraga-byte/versaoteste.git"
WORK_DIR = "/tmp/pesquisai"
PANEL_ID = "ufvai_boot_panel"

_CSS = """<style>
.ufl-panel{max-width:520px;margin:8px auto;padding:20px 24px;background:#f6f5f0;border:1px solid rgba(43,45,58,.14);border-radius:16px;font-family:system-ui,sans-serif}
.ufl-wordmark{font-size:30px;font-weight:800;text-align:center;color:#2b2d3a}
.ufl-wordmark .ufl-ai{color:#b8912f}
.ufl-tagline{font-size:10px;letter-spacing:.35em;text-align:center;color:#8f8d86;margin-top:4px}
.ufl-barwrap{height:10px;background:#e9e6df;border-radius:999px;overflow:hidden;margin:14px 0 6px}
.ufl-fill{height:100%;width:PCT%;background:linear-gradient(90deg,#caa53f,#8f7536);border-radius:999px;transition:width .45s}
.ufl-pct{text-align:right;font-size:11px;color:#8f7536}
.ufl-msg{font-size:13px;color:#3c3f4e;margin-top:8px}
</style>"""


def _panel(pct, msg):
    return HTML(_CSS.replace("PCT", str(pct)) +
                '<div class="ufl-panel"><div class="ufl-wordmark"><span>UFV</span>'
                '<span class="ufl-ai">AI</span></div>'
                '<div class="ufl-tagline">INTELIGÊNCIA ARTIFICIAL</div>'
                '<div class="ufl-barwrap"><div class="ufl-fill"></div></div>'
                f'<div class="ufl-pct">{pct}%</div>'
                f'<div class="ufl-msg">{msg}</div></div>')


def _autopilot_ok():
    return subprocess.run(
        [sys.executable, "-c",
         "from pesquisai.obsidian.autopilot import recall, save"],
        capture_output=True, cwd="/").returncode == 0


def _origin(path):
    try:
        return subprocess.run(["git", "-C", path, "remote", "get-url", "origin"],
                              capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        return ""


def _tree_ok(path):
    """Clone válido = mesma origem do REPO_URL, com ocweb.py (UI web) e versão 0.7.11."""
    if not os.path.isdir(os.path.join(path, "pesquisai")):
        return False
    if _origin(path).rstrip("/") != REPO_URL.rstrip("/"):
        return False
    if not os.path.isfile(os.path.join(path, "pesquisai", "ocweb.py")):
        return False
    try:
        v = open(os.path.join(path, "pesquisai", "__version__.py"), encoding="utf-8").read()
        return '__version__: str = "0.7.11"' in v
    except OSError:
        return False


def _sync_repo():
    """NUNCA reaproveita /tmp/pesquisai de outro repositório/versão.

    O bug que deixava o ttyd no ar: se /tmp/pesquisai já existia (clone do
    repositório antigo), o `git pull` puxava a 0.7.10 daquela origem e o
    boot rodava código sem `ocweb.py`.
    """
    if _tree_ok(WORK_DIR):
        subprocess.run(["git", "-C", WORK_DIR, "pull", "--ff-only", "--depth", "1"],
                       capture_output=True)
        return "reutilizado"
    shutil.rmtree(WORK_DIR, ignore_errors=True)
    res = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK_DIR],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise SystemExit(f"[ERRO] git clone falhou:\n{res.stderr[-800:]}")
    return "clonado"


def _porta_aberta(porta):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", porta)) == 0


display(_panel(5, "Baixando o UFVAI (origem correta)..."), display_id=PANEL_ID)
_como = _sync_repo()
if not _tree_ok(WORK_DIR):
    raise SystemExit("[ERRO] a árvore baixada não tem a UI web (ocweb.py) ou a versão não é 0.7.11")

_version = open(os.path.join(WORK_DIR, "pesquisai", "__version__.py"), encoding="utf-8").read()
_version = _version.split('__version__: str = "')[1].split('"')[0]
print(f"UFVAI {_version} — árvore {_com} em {WORK_DIR}")

update_display(_panel(20, "Instalando o pacote `pesquisai` (API do autopilot)..."),
               display_id=PANEL_ID)

for _cmd in (
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK_DIR],
    [sys.executable, "-m", "pip", "install", "-q", "-e", WORK_DIR],
    [sys.executable, "-m", "pip", "install", "-q", "--user",
     "--break-system-packages", "-e", WORK_DIR],
):
    try:
        subprocess.run(_cmd, capture_output=True, text=True, timeout=900)
    except Exception:
        pass
    if _autopilot_ok():
        break
else:
    print("[AVISO] pacote `pesquisai` nao instalado; a API do autopilot fica "
          "indisponivel fora de /tmp/pesquisai")

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# O servidor do wrapper é uma thread do kernel: se a porta 8001 já responde,
# é a interface ANTIGA desta sessão. Não há como substituí-la sem reiniciar.
if _porta_aberta(8001):
    raise SystemExit(
        "\n⚠️  A porta 8001 já está em uso (interface antiga desta sessão).\n"
        "   Faça **Ambiente de execução → Reiniciar sessão** e rode esta célula\n"
        "   novamente. Sem reiniciar, o ttyd continua sendo servido.\n"
    )

update_display(_panel(60, "Iniciando o UFVAI (interface web)..."), display_id=PANEL_ID)

from main import run
run()

update_display(_panel(90, "Ativando o keep-alive..."), display_id=PANEL_ID)

keepalive = subprocess.Popen(
    [sys.executable, "-u", "-c",
     "import time, datetime\n"
     "while True:\n"
     "    print('keepalive', datetime.datetime.now(), flush=True)\n"
     "    time.sleep(60)\n"],
    cwd=WORK_DIR,
    stdin=subprocess.DEVNULL,
    stdout=open(os.path.join(WORK_DIR, "ufvai_keepalive.log"), "ab"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

with open(os.path.join(WORK_DIR, "ufvai_keepalive.pid"), "w") as _kf:
    _kf.write(str(keepalive.pid))

update_display(_panel(100, f"UFVAI Carregado. Keep-alive ativo (PID {keepalive.pid})."),
               display_id=PANEL_ID)


# ---

**Citação ABNT:**

```
BRAGA, Gustavo Bastos. UFVAI: agente de inteligência artificial para
pesquisa científica. Versão 0.7.11. Viçosa: Universidade Federal de Viçosa,
2026. Disponível em: https://colab.research.google.com/github/
gustavobraga-byte/versaoteste/. Projeto SisPPG/UFV nº 10356285004.
```

👨‍💻 **Gustavo Bastos Braga** (UFV) — [gustavo.braga@ufv.br](mailto:gustavo.braga@ufv.br)  
⭐ Código: [github.com/gustavobraga-byte/versaoteste](https://github.com/gustavobraga-byte/versaoteste)  
📜 MIT · [Termos v3](https://github.com/gustavobraga-byte/versaoteste/blob/main/docs/TERMS_OF_USE.md) · [Privacidade v1.1](https://github.com/gustavobraga-byte/versaoteste/blob/main/PRIVACY.md) · [Telemetria](https://github.com/gustavobraga-byte/versaoteste/blob/main/TELEMETRY.md) · [🌐 Site oficial](https://der.ufv.br/ufvai/)
